# Implementing `Co-Clustering Triples from Open Information Extraction` From Scratch

### Setting Up Library

In [3]:
import torch
import random
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from itertools import combinations
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.cluster import AgglomerativeClustering

### 1. Data Simulation and Embedding Generation

In [4]:
def simulate_numerical_data():
    base_facts = [
        {'id': 1, 's': 'India', 'p': 'has a population of', 'o': 1400000000},
        {'id': 2, 's': 'Mount Everest', 'p': 'has a height of', 'o': 8848},
    ] # Simulated numerical data
    
    generated_facts = list(base_facts)
    aliases = {
        'India': ['Bharat', 'Hindustan'],
        'Mount Everest': ['Sagarmatha', 'Everest'],
    }
    pred_phrases = {'has a population of': 'population stands at',
                    'has a height of': 'is tall'}# predicted phrases
    for fact in base_facts:
        new_fact = fact.copy()# copying the fact
        new_fact['s'] = aliases.get(fact['s'], [fact['s']]) # randomly selecting an alias
        new_fact['p'] = pred_phrases.get(fact['p'], fact['p']) # using predicted phrase
        variation = fact['o'] * random.uniform(-0.1, 0.1) # adding variation to the object for numerical data
        new_fact['o'] = int(fact['o'] + variation) # updating the object with variation
        generated_facts.append(new_fact) # appending the new fact to the list
    generated_facts.extend([
        {'id': 3, 's': 'Brazil', 'p': 'has a GDP of', 'o': 1600000000000},
        {'id': 4, 's': 'K2', 'p': 'is located in', 'o': 8611},
    ])# adding more facts
    print("Simulated numerical dataset.")
    return generated_facts
        

- #### Building Embedding Generation

In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')# loading the tokenizer
bert_model = BertModel.from_pretrained('bert-base-uncased')# loading the BERT model
BERT_DIM = bert_model.config.hidden_size # getting the BERT dimension